# 02. 形態学的解析（Morphological Analysis） — 練習問題

**対象技術**: 量子コンピューティング

問題を相互に独立なパラメータ（次元）に分解し、各パラメータの取りうる値の全組み合わせ（可能性空間）を生成する。次に「整合性マトリクス」で両立しない値ペアを含む構成を除外し、ありうる技術構成だけを残す。残った整合構成は Futures Wheel やシナリオ・プランニングの種になる。

`itertools` で全構成を生成し、`numpy` / `matplotlib` で整合性マトリクスを可視化する。

In [ ]:
import itertools
import numpy as np
import matplotlib.pyplot as plt
%matplotlib inline

# --- 日本語フォント設定: 共通モジュール jp_font.py を読み込む ---
# フォント探索・登録・フォールバックの実装は repo 直下の jp_font.py に集約。
import os as _os, sys as _sys
_d = _os.path.abspath(_os.getcwd())
while not _os.path.exists(_os.path.join(_d, "jp_font.py")) and _d != _os.path.dirname(_d):
    _d = _os.path.dirname(_d)
_sys.path.insert(0, _d)
from jp_font import setup_japanese_font
setup_japanese_font()


## 形態ボックスの定義

量子コンピューティングの発展形態を 5 つのパラメータで定義する。各パラメータは相互に独立（直交）であることが望ましい。

In [ ]:
PARAMETERS = {
    "量子ビット方式": ["超伝導", "イオントラップ", "光", "中性原子"],
    "主用途":         ["暗号解読", "組合せ最適化", "量子化学", "量子ML"],
    "提供形態":       ["クラウド", "オンプレ", "ハイブリッド"],
    "成熟度":         ["NISQ", "FTQC"],
    "主担い手":       ["国家", "巨大IT", "専業スタートアップ"],
}

print("形態ボックス(パラメータ x 値):")
for p, vals in PARAMETERS.items():
    print(f"  {p:<14}: {' / '.join(vals)}")

## 整合性マトリクス

両立しない値のペア（非整合ペア）を `frozenset` で順不同に定義する。判定理由も併記する。

In [ ]:
INCOMPATIBLE = [
    (frozenset({"暗号解読", "NISQ"}),
     "Shorで RSA-2048 を破るには誤り耐性計算が必須。NISQ では不可能。"),
    (frozenset({"光", "オンプレ"}),
     "光量子計算は専用安定化環境を要し、汎用顧客のオンプレ設置は非現実的。"),
    (frozenset({"専業スタートアップ", "FTQC"}),
     "FTQC 構築の資本規模は専業スタートアップ単独では釣り合わない。"),
    (frozenset({"量子ML", "オンプレ"}),
     "量子ML はクラウド上の大規模データ連携が前提で、孤立オンプレと相反。"),
    (frozenset({"中性原子", "オンプレ"}),
     "中性原子方式はレーザ系が大規模で顧客オンプレ設置が困難。"),
]
print(f"非整合ペア: {len(INCOMPATIBLE)} 件")

## 解析関数

構成が全整合性制約を満たすか判定する関数と、全パラメータの直積で全構成を生成する関数を定義する。

In [ ]:
def is_consistent(config):
    """構成(値の組)が全ての整合性制約を満たすか判定する。"""
    values = set(config.values())
    for pair, _reason in INCOMPATIBLE:
        if pair <= values:  # 非整合ペアが構成に丸ごと含まれる
            return False
    return True


def generate_all_configs():
    """全パラメータの直積で全構成を生成する。"""
    keys = list(PARAMETERS.keys())
    value_lists = [PARAMETERS[k] for k in keys]
    configs = []
    for combo in itertools.product(*value_lists):
        configs.append(dict(zip(keys, combo)))
    return configs

## 全構成の生成と整合性フィルタ

全構成を生成し、非整合ペアでフィルタして整合構成数と整合率を求める。

In [ ]:
all_configs = generate_all_configs()
total = len(all_configs)
sizes = [len(v) for v in PARAMETERS.values()]
size_expr = " x ".join(str(s) for s in sizes)
print(f"全構成数 = {size_expr} = {total} 通り")

consistent = [c for c in all_configs if is_consistent(c)]
n_ok = len(consistent)
rate = n_ok / total
print(f"整合構成数 = {n_ok} 通り")
print(f"整合率     = {n_ok}/{total} = {rate:.3f}")

## 非整合ペアごとの除外内訳

各非整合ペアが何構成を除外対象とするか（重複あり）を集計する。

In [ ]:
print("非整合ペアごとの該当構成数(重複あり):")
for pair, reason in INCOMPATIBLE:
    cnt = sum(1 for c in all_configs if pair <= set(c.values()))
    label = " & ".join(sorted(pair))
    print(f"  [{label}] {cnt} 構成を除外")
    print(f"     理由: {reason}")

## 整合構成内での値の出現頻度

整合構成のなかで各パラメータ値がどれだけ残りやすいかを集計し、サンプル構成も表示する。

In [ ]:
print("整合構成内での各値の出現頻度:")
for p, vals in PARAMETERS.items():
    counts = {v: sum(1 for c in consistent if c[p] == v) for v in vals}
    line = "  ".join(f"{v}:{counts[v]}" for v in vals)
    print(f"  {p:<14}: {line}")

print()
print("整合構成のサンプル(先頭3件):")
for c in consistent[:3]:
    desc = " / ".join(c[p] for p in PARAMETERS)
    print(f"  - {desc}")

print()
print("[解釈] 整合性チェックにより『暗号解読 x FTQC x 国家』のような")
print("       警戒すべき構成が、明示的に評価対象として浮かび上がる。")

## 可視化: 整合性マトリクスと構成数比較

左図は値ペアごとの整合性をヒートマップ表示する（赤＝非整合、青＝整合）。右図は全構成数と整合構成数を棒グラフで比較する。

In [ ]:
# 全パラメータ値を一列に並べる
all_values = []
value_param = []
for p, vals in PARAMETERS.items():
    for v in vals:
        all_values.append(v)
        value_param.append(p)
m = len(all_values)

# 整合性マトリクス: 1=整合, 0=非整合, 対角は無効(0.5)
incompat_pairs = set()
for pair, _ in INCOMPATIBLE:
    a, b = sorted(pair)
    incompat_pairs.add((a, b))

M = np.ones((m, m))
for i in range(m):
    for j in range(m):
        if i == j or value_param[i] == value_param[j]:
            M[i, j] = 0.5  # 同一パラメータ内 / 対角は判定対象外
        else:
            a, b = sorted([all_values[i], all_values[j]])
            if (a, b) in incompat_pairs:
                M[i, j] = 0.0  # 非整合

fig, axes = plt.subplots(1, 2, figsize=(15, 7),
                         gridspec_kw={"width_ratios": [1.5, 1]})

# 左: 整合性マトリクス ヒートマップ
ax = axes[0]
from matplotlib.colors import ListedColormap
cmap = ListedColormap(["#d1495b", "#cccccc", "#3a6ea5"])
im = ax.imshow(M, cmap=cmap, vmin=0, vmax=1)
labels = [f"{i}" for i in range(m)]
ax.set_xticks(range(m))
ax.set_yticks(range(m))
ax.set_xticklabels(labels, fontsize=8)
ax.set_yticklabels(labels, fontsize=8)
ax.set_title("Consistency Matrix\n(red=incompatible, blue=compatible, grey=N/A)",
             fontsize=11)
ax.set_xlabel("value index")
ax.set_ylabel("value index")

# 値インデックスの凡例をテキストで添える
note = "\n".join(f"{i}: {all_values[i]}" for i in range(m))
ax.text(m + 0.6, 0, note, fontsize=7, va="top", ha="left")

# 右: 全構成数 vs 整合構成数 棒グラフ
ax2 = axes[1]
bars = ax2.bar(["All configs", "Consistent configs"], [total, n_ok],
               color=["#9aa0a6", "#3a6ea5"], width=0.55)
for b, val in zip(bars, [total, n_ok]):
    ax2.text(b.get_x() + b.get_width() / 2, val + 4, str(val),
             ha="center", fontsize=11, fontweight="bold")
ax2.set_ylabel("number of configurations")
ax2.set_title(f"Possibility Space\nconsistency rate = {rate:.3f}",
              fontsize=11)
ax2.set_ylim(0, total * 1.15)

plt.tight_layout()
plt.show()

## 代表構成のクラスタリング抽出

整合構成は数十〜数百通り残り、そのまま全件をシナリオ化するのは現実的でない。そこで整合構成集合を「互いに似た構成のまとまり（クラスタ）」に分け、各クラスタの代表となる典型構成だけを抽出する。

- **構成間の距離**: ハミング距離（一致しないパラメータ値の個数）を用いる。全パラメータが一致すれば 0、すべて異なれば パラメータ数に等しい。
- **手法**: k-medoids 法を自前実装する。k-means と異なり、クラスタ代表（medoid）を**実在の整合構成のなかから**選ぶため、代表が必ず整合性制約を満たす実在シナリオになる。これは形態学的解析と相性が良い。


In [ ]:
# --- 構成間のハミング距離 ---
_keys = list(PARAMETERS.keys())

def hamming(c1, c2):
    """2 構成間のハミング距離: 値が一致しないパラメータの個数。"""
    return sum(1 for k in _keys if c1[k] != c2[k])

# 整合構成集合の全ペア距離行列を事前計算する
n = len(consistent)
D = np.zeros((n, n), dtype=int)
for i in range(n):
    for j in range(i + 1, n):
        d = hamming(consistent[i], consistent[j])
        D[i, j] = d
        D[j, i] = d

print(f"整合構成数 n = {n}")
print(f"距離行列 D の形状 = {D.shape}, 取りうる距離 = 0..{len(_keys)}")


# --- k-medoids 法の自前実装 (PAM 風: medoid 入替で総コストを最小化) ---
def k_medoids(D, k, max_iter=100, seed=0):
    """事前計算した距離行列 D に対し k-medoids クラスタリングを行う。
    戻り値: (medoid のインデックス配列, 各点のクラスタ割当配列)。"""
    rng = np.random.RandomState(seed)
    n = D.shape[0]
    # 初期 medoid: 重複なしランダム選択
    medoids = list(rng.choice(n, size=k, replace=False))

    def assign(medoids):
        # 各点を最も近い medoid のクラスタへ割り当てる
        return np.argmin(D[:, medoids], axis=1)

    def total_cost(medoids, labels):
        return sum(D[i, medoids[labels[i]]] for i in range(n))

    labels = assign(medoids)
    cost = total_cost(medoids, labels)
    for _ in range(max_iter):
        improved = False
        for ci in range(k):
            members = np.where(labels == ci)[0]
            # クラスタ内で総距離が最小の点を新 medoid 候補にする
            for cand in members:
                if cand == medoids[ci]:
                    continue
                trial = list(medoids)
                trial[ci] = cand
                trial_labels = assign(trial)
                trial_cost = total_cost(trial, trial_labels)
                if trial_cost < cost:
                    medoids, labels, cost = trial, trial_labels, trial_cost
                    improved = True
        if not improved:
            break
    return np.array(medoids), labels


# k は 4〜6 程度。ここでは 5 とする。
# 根拠: 整合構成を「典型シナリオ」として扱える程度に集約しつつ、
#       後続のシナリオ・プランニングで比較しやすい少数(両手で数えられる)に収めるため。
K = 5
np.random.seed(42)  # 初期 medoid 選択の乱数を固定し再現性を確保する
medoid_idx, labels = k_medoids(D, K, seed=42)

cluster_sizes = [int(np.sum(labels == c)) for c in range(K)]
print(f"\nk = {K} でクラスタリングした。")
print(f"クラスタサイズ = {cluster_sizes} (合計 {sum(cluster_sizes)})")


In [ ]:
# --- 各クラスタの medoid(代表構成)を人間可読に出力する ---
# サイズの大きいクラスタから順に並べる
order = sorted(range(K), key=lambda c: cluster_sizes[c], reverse=True)

print("=" * 56)
print(" 代表構成（各クラスタの medoid = 典型シナリオ）")
print("=" * 56)
for rank, c in enumerate(order, start=1):
    rep = consistent[medoid_idx[c]]
    print(f"\n[クラスタ{rank}]  サイズ {cluster_sizes[c]} 構成")
    for p in _keys:
        print(f"    {p:<14}: {rep[p]}")

# 代表構成どうしのハミング距離（性格の異なり具合）も確認する
print("\n" + "-" * 56)
print(" 代表構成間のハミング距離（大きいほど性格が異なる）")
print("-" * 56)
for a in range(K):
    for b in range(a + 1, K):
        ra, rb = order[a], order[b]
        d = D[medoid_idx[ra], medoid_idx[rb]]
        print(f"  クラスタ{a+1} <-> クラスタ{b+1}: {d}")


In [ ]:
# --- 可視化: 距離行列を medoid 順に並べた imshow ヒートマップ + クラスタサイズ棒グラフ ---
# クラスタ単位で構成を並べ替えると、クラスタが対角ブロックとして浮かび上がる。
perm = []
block_bounds = []  # クラスタ境界の位置
for c in order:
    members = list(np.where(labels == c)[0])
    # クラスタ内は medoid からの距離順に並べる
    members.sort(key=lambda i: D[i, medoid_idx[c]])
    perm.extend(members)
    block_bounds.append(len(perm))
perm = np.array(perm)
D_sorted = D[np.ix_(perm, perm)]

fig, axes = plt.subplots(1, 2, figsize=(14, 6),
                         gridspec_kw={"width_ratios": [1.2, 1]})

# 左: クラスタ順に並べた距離行列ヒートマップ
ax = axes[0]
im = ax.imshow(D_sorted, cmap="viridis_r", aspect="auto")
ax.set_title("Hamming distance matrix\n(sorted by cluster -> diagonal blocks)",
             fontsize=11)
ax.set_xlabel("consistent config (cluster-sorted)")
ax.set_ylabel("consistent config (cluster-sorted)")
# クラスタ境界線を引く
prev = 0
for b in block_bounds[:-1]:
    ax.axhline(b - 0.5, color="white", lw=1.2)
    ax.axvline(b - 0.5, color="white", lw=1.2)
# 各ブロック中央にクラスタ番号
prev = 0
for rank, b in enumerate(block_bounds, start=1):
    mid = (prev + b) / 2 - 0.5
    ax.text(mid, mid, f"C{rank}", color="white", fontsize=12,
            fontweight="bold", ha="center", va="center")
    prev = b
fig.colorbar(im, ax=ax, fraction=0.046, pad=0.04, label="Hamming distance")

# 右: クラスタサイズ棒グラフ
ax2 = axes[1]
sizes_ordered = [cluster_sizes[c] for c in order]
xlabels = [f"C{r}" for r in range(1, K + 1)]
bars = ax2.bar(xlabels, sizes_ordered, color="#3a6ea5", width=0.6)
for bar, val in zip(bars, sizes_ordered):
    ax2.text(bar.get_x() + bar.get_width() / 2, val + 0.4, str(val),
             ha="center", fontsize=11, fontweight="bold")
ax2.set_ylabel("number of configurations")
ax2.set_title(f"Cluster sizes (k = {K})", fontsize=11)
ax2.set_ylim(0, max(sizes_ordered) * 1.18)

plt.tight_layout()
plt.show()


### 代表構成抽出の意義

抽出された 5 つの代表構成（medoid）は、`itertools.product` で列挙し整合性で絞り込んだ**網羅的な可能性空間から選ばれた、少数の典型シナリオ**である。ヒートマップで対角ブロックがはっきり分かれていれば、それぞれのクラスタが互いに性格の異なる「未来の塊」をうまく代表できていることを意味する。

これら少数の代表構成は、

- **シナリオ・プランニング**: 各代表構成を 1 本のシナリオの骨格として展開する、
- **Futures Wheel**: 各代表構成を中心ノードに置き、波及的な影響を放射状に展開する、

といった後続手法の評価対象（入力）になる。数百通りの整合構成を全件扱う代わりに、互いに離れた少数の典型に絞ることで、議論を網羅性を保ったまま現実的な規模へ落とし込める。


## 未来デザイン論文での使われ方と結論への影響

形態学的解析は、未来デザイン論文において、対象技術を構成するパラメータとその取りうる値を体系的に列挙し、それらの組み合わせが張る「可能性空間」全体を提示する装置として用いられる。論文は典型的に、技術・制度・社会の各次元を軸に設定し、各軸の選択肢を掛け合わせて全構成を生成したうえで、内部矛盾を含む構成を整合性チェックで除外し、残った整合的構成の集合と、そこから抽出した代表シナリオを示す。主眼は単一の予測ではなく、検討に値する選択肢を漏れなく卓上に並べることにある。

この手法がもたらす結論の型は「可能性空間のマッピング」である。とりわけ価値があるのは、直感や既存の議論では結びつけられてこなかったパラメータの組み合わせ——これまで見落とされていた構成——を機械的な全列挙によって浮かび上がらせる点であり、結論は「未来はこうなる」ではなく「検討すべき未来の幅はここまで広い」という形をとる。時間観の経路で見れば、未来は過去の延長としてではなく、構成の選択肢の集合として描かれ、どの構成を選ぶかという選択の余地が結論の中心に据えられる。境界設定の経路では、結論の網羅性は軸の選び方と各軸の値の粒度に全面的に依存し、軸に立てられなかった次元は可能性空間に存在しないものとして扱われる。

同時に、この手法には固有の限界がある。形態学的解析はパラメータ次元を互いに独立とみなして組み合わせを生成するため、現実には次元間の相互依存が強い領域では、整合性チェックで一部を除いてもなお構成空間を過大に見積もりやすい。実現可能性が極端に異なる構成が同じ平面上に等しく並ぶことで、論文が選択肢の豊かさを強調するあまり、各構成の蓋然性の差を結論が捨象してしまう危険もある。この手法を採る論文の信頼性は、軸の設定根拠と整合性ルールの透明性、そして構成間の重みづけをどう扱うかの明示にかかっている。

## 発展課題

**課題A**: 整合構成間のハミング距離（一致しないパラメータ数）を定義し、k-medoids 風の自前クラスタリングを実装して代表構成を抽出せよ。性格の異なる代表構成がシナリオの種になる。

**課題B**: 新パラメータ（例: 量子誤り訂正符号 = 表面符号 / LDPC符号 / なし）を`PARAMETERS` に追加し、全構成数と整合率がどう変わるか観察せよ。必要なら `INCOMPATIBLE` に新たな非整合ペアも加えること。